# 00 — Environment Check

**Purpose:** Verify that all required packages, Python version, and system dependencies are correctly installed before running the pipeline.

**Inputs:** None

**Outputs:** Printed confirmation of package versions and environment readiness

![Status: COMPLETE](https://img.shields.io/badge/Status-COMPLETE-brightgreen)

In [1]:
# ── Cell 1: Install all required packages ─────────────────────────────────────
import subprocess, sys

PACKAGES = [
    "wfdb",
    "neurokit2",
    "biosppy",
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "xgboost",
    "matplotlib",
    "seaborn",
    "joblib",
    "tableauhyperapi",
    "pantab",
    "ipywidgets",
    "peakutils"
]

print("Installing packages...\n")
for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", pkg],
        capture_output=True, text=True
    )
    status = "OK" if result.returncode == 0 else f"FAILED: {result.stderr.strip()[:80]}"
    print(f"  {pkg:<20} {status}")

print("\nInstallation step complete.")

Installing packages...

  wfdb                 OK
  neurokit2            OK
  biosppy              OK
  numpy                OK
  pandas               OK
  scipy                OK
  scikit-learn         OK
  xgboost              OK
  matplotlib           OK
  seaborn              OK
  joblib               OK
  tableauhyperapi      OK
  pantab               OK
  ipywidgets           OK
  peakutils            OK

Installation step complete.


In [2]:
# ── Cell 2: Import packages and collect versions ───────────────────────────────
import sys, importlib

# Map pip/install name → importable module name
IMPORT_MAP = {
    "wfdb":            "wfdb",
    "neurokit2":       "neurokit2",
    "biosppy":         "biosppy",
    "numpy":           "numpy",
    "pandas":          "pandas",
    "scipy":           "scipy",
    "scikit-learn":    "sklearn",
    "xgboost":         "xgboost",
    "matplotlib":      "matplotlib",
    "seaborn":         "seaborn",
    "joblib":          "joblib",
    "tableauhyperapi": "tableauhyperapi",
    "pantab":          "pantab",
    "ipywidgets":      "ipywidgets",
}

import_results = {}  # pkg_name → (module | None, version_str | error_str)

for pkg, mod_name in IMPORT_MAP.items():
    try:
        mod = importlib.import_module(mod_name)
        ver = getattr(mod, "__version__", "n/a")
        import_results[pkg] = (mod, ver)
    except Exception as exc:
        import_results[pkg] = (None, f"IMPORT ERROR: {exc}")

print(f"Python {sys.version}\n")

Python 3.13.2 (v3.13.2:4f8bb3947cf, Feb  4 2025, 11:51:10) [Clang 15.0.0 (clang-1500.3.9.4)]



In [3]:
# ── Cell 3: Print formatted version table ─────────────────────────────────────
col_w = 20
print(f"{'Package':<{col_w}} {'Version':<15} {'Status'}")
print("-" * (col_w + 30))
for pkg, (mod, ver) in import_results.items():
    ok = mod is not None
    status = "PASS" if ok else "FAIL"
    print(f"{pkg:<{col_w}} {ver:<15} {status}")

Package              Version         Status
--------------------------------------------------
wfdb                 4.3.1           PASS
neurokit2            0.2.13          PASS
biosppy              2.1.2           PASS
numpy                2.2.6           PASS
pandas               2.3.1           PASS
scipy                1.16.1          PASS
scikit-learn         1.8.0           PASS
xgboost              3.2.0           PASS
matplotlib           3.10.7          PASS
seaborn              0.13.2          PASS
joblib               1.5.3           PASS
tableauhyperapi      0.0.24457       PASS
pantab               5.3.0           PASS
ipywidgets           8.1.8           PASS


In [4]:
# ── Cell 4: Tableau Hyper API functional check ─────────────────────────────────
# Creates an in-memory Hyper file, writes one row, reads it back, then deletes it.

import os, pathlib

tableau_pass = False
tableau_error = ""

try:
    from tableauhyperapi import (
        HyperProcess, Telemetry, Connection, CreateMode,
        TableDefinition, SqlType, TableName, Inserter
    )

    hyper_path = pathlib.Path("/tmp/_env_check_test.hyper")
    hyper_path.unlink(missing_ok=True)

    # Start Hyper process and write one row
    with HyperProcess(telemetry=Telemetry.DO_NOT_SEND_USAGE_DATA_TO_TABLEAU) as hyper:
        with Connection(
            endpoint=hyper.endpoint,
            database=str(hyper_path),
            create_mode=CreateMode.CREATE_AND_REPLACE
        ) as conn:
            table_def = TableDefinition(
                table_name=TableName("public", "test"),
                columns=[
                    TableDefinition.Column("id",  SqlType.int()),
                    TableDefinition.Column("val", SqlType.text()),
                ]
            )
            conn.catalog.create_table(table_def)

            with Inserter(conn, table_def) as inserter:
                inserter.add_row([1, "hello-hyper"])
                inserter.execute()

            # Read back
            rows = conn.execute_list_query("SELECT id, val FROM public.test")

    assert len(rows) == 1 and rows[0][0] == 1 and rows[0][1] == "hello-hyper", \
        f"Unexpected result: {rows}"

    hyper_path.unlink(missing_ok=True)
    tableau_pass = True
    print("Tableau Hyper API check: row written and read back successfully.")
    print(f"  Result: {rows}")

except Exception as exc:
    tableau_error = str(exc)
    print(f"Tableau Hyper API check FAILED: {tableau_error}")

Tableau Hyper API check: row written and read back successfully.
  Result: [[1, 'hello-hyper']]


In [5]:
# ── Cell 5: PASS / FAIL summary ───────────────────────────────────────────────
all_pass = True

print("=" * 55)
print(" ENVIRONMENT CHECK SUMMARY")
print("=" * 55)
print(f"  {'Check':<30} {'Result'}")
print("-" * 55)

for pkg, (mod, ver) in import_results.items():
    ok = mod is not None
    label = "PASS" if ok else "FAIL"
    if not ok:
        all_pass = False
    detail = ver if ok else ver  # ver already contains error msg on failure
    print(f"  {pkg:<30} {label}  ({detail})")

print("-" * 55)
hyper_label = "PASS" if tableau_pass else "FAIL"
if not tableau_pass:
    all_pass = False
hyper_detail = "in-memory write/read OK" if tableau_pass else tableau_error[:50]
print(f"  {'Tableau Hyper API (functional)':<30} {hyper_label}  ({hyper_detail})")
print("=" * 55)

if all_pass:
    print("\n>>> ALL CHECKS PASSED. Ready to proceed to notebook 01. <<<")
else:
    print("\n>>> ONE OR MORE CHECKS FAILED. Resolve issues before continuing. <<<")

 ENVIRONMENT CHECK SUMMARY
  Check                          Result
-------------------------------------------------------
  wfdb                           PASS  (4.3.1)
  neurokit2                      PASS  (0.2.13)
  biosppy                        PASS  (2.1.2)
  numpy                          PASS  (2.2.6)
  pandas                         PASS  (2.3.1)
  scipy                          PASS  (1.16.1)
  scikit-learn                   PASS  (1.8.0)
  xgboost                        PASS  (3.2.0)
  matplotlib                     PASS  (3.10.7)
  seaborn                        PASS  (0.13.2)
  joblib                         PASS  (1.5.3)
  tableauhyperapi                PASS  (0.0.24457)
  pantab                         PASS  (5.3.0)
  ipywidgets                     PASS  (8.1.8)
-------------------------------------------------------
  Tableau Hyper API (functional) PASS  (in-memory write/read OK)

>>> ALL CHECKS PASSED. Ready to proceed to notebook 01. <<<
